# Lab Day 2 — chay mot mach tren Colab (DeepWeeds)

Notebook nay chay **toan bo lab** tu dau den cuoi. Ban chi can:

1. Runtime > Change runtime type > **GPU (T4)**.

2. Data: **cach A**: bo qua muc 0b, de muc 1 tu tai Zenodo. **cach B**: up `data.zip` len Drive (vd `MyDrive/data.zip`), chay muc 0b de copy ve.

3. Runtime > Run all. Ket qua (runs/, predictions/, curves/, *.xlsx) tai ve bang cell cuoi.

Luu y: `FULL=False` chay khoi dong nhanh (2 epoch); doi `FULL=True` de chay that (12 epoch, ~20-25 lan train).

## 0. Cai dat + duong dan

In [ ]:
%pip -q install timm openpyxl
import sys, os, platform
sys.path.insert(0, "code")
sys.path.insert(0, ".")
import torch
try:
    import timm; print("timm", timm.__version__)
except ImportError:
    print("THIEU timm")
print("python", platform.python_version(), "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU -> doi Runtime sang GPU!")
try:
    import eval as ev; print("eval.py OK")
except ImportError:
    print("THIEU eval.py: upload file eval.py (goc repo) vao cung thu muc voi code/ roi chay lai")

## 0b. Lay data.zip tu Drive (tuy chon — bo qua neu de muc 1 tu tai)

Up `data.zip` len Drive truoc (vd `MyDrive/data.zip`), chay cell duoi de mount + copy ve. Neu khong, bo qua cell nay.

In [ ]:
import os, glob, shutil
DRIVE_ZIP = "/content/drive/MyDrive/data.zip"  # DOI neu ban de cho khac
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive OK")
except ImportError:
    print("Khong phai Colab. Bo qua mount.")
print("Kiem tra:", DRIVE_ZIP, os.path.exists(DRIVE_ZIP))
if os.path.exists(DRIVE_ZIP):
    print("Copy tu Drive ve...")
    shutil.copy(DRIVE_ZIP, "data.zip")
    print("Giai nen data.zip ...")
    shutil.unpack_archive("data.zip", ".")
elif os.path.exists("data.zip"):
    print("Giai nen data.zip co san ...")
    shutil.unpack_archive("data.zip", ".")
else:
    print("Khong thay file tren Drive. Bo qua (muc 1 se tu tai).")
    print("Neu muon upload truc tiep: from google.colab import files; files.upload()")
jpgs0 = glob.glob("data/**/*.jpg", recursive=True)
print("So jpg hien co trong data/:", len(jpgs0))
print(sorted(os.listdir("data"))[:10] if os.path.exists("data") else "chua co data/")
print(sorted(os.listdir("data/labels")) if os.path.exists("data/labels") else "chua co data/labels")


## 1. Tai du lieu + tim IMAGES_DIR tu dong (tu bo qua neu da upload du)

In [ ]:
import hashlib, glob, urllib.request, shutil
os.makedirs("data/labels", exist_ok=True)
jpgs_exist = glob.glob("data/**/*.jpg", recursive=True)
if len(jpgs_exist) >= 17000:
    print("Da co san", len(jpgs_exist), "anh (do upload) -> bo qua tai Zenodo")
else:
    print("Chua du anh (%d) -> tai tu Zenodo..." % len(jpgs_exist))
    if not os.path.exists("data/images.zip"):
        urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", "data/images.zip")
    h = hashlib.md5()
    with open("data/images.zip", "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", "MD5 sai: " + h.hexdigest()
    print("MD5 OK")
    shutil.unpack_archive("data/images.zip", "data/")
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    dst = "data/labels/" + name + ".csv"
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE + "/" + name + ".csv", dst)
print(sorted(os.listdir("data/labels")))
from collections import Counter
jpgs = glob.glob("data/**/*.jpg", recursive=True)
assert jpgs, "khong thay jpg trong data/"
IMAGES_DIR = Counter(os.path.dirname(j) for j in jpgs).most_common(1)[0][0]
LABELS_DIR = "data/labels"
print("IMAGES_DIR =", IMAGES_DIR, "| so jpg =", len(jpgs))

## 2. Buoc 0 — EDA + kiem tra split (bat buoc)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
import dataset as D
tr_df, va_df, te_df = D.load_split(LABELS_DIR, 0)
info = D.check_split(tr_df, va_df, te_df, IMAGES_DIR)
print(info["n"], "total =", info["total"])
names = ["Chinee apple","Lantana","Parkinsonia","Parthenium","Prickly acacia","Rubber vine","Siam weed","Snake weed","Negative"]
tot = [info["per_class"]["train"].get(i,0)+info["per_class"]["val"].get(i,0)+info["per_class"]["test"].get(i,0) for i in range(9)]
paper = [1125,1064,1031,1022,1062,1009,1074,1016,9106]
for i,n in enumerate(names): print(n, "dem =", tot[i], "| bao =", paper[i])
pd.DataFrame({"train": [info["per_class"]["train"].get(i,0) for i in range(9)],
              "val": [info["per_class"]["val"].get(i,0) for i in range(9)],
              "test": [info["per_class"]["test"].get(i,0) for i in range(9)]}, index=names).plot.bar(figsize=(10,4))
plt.title("Phan bo lop train/val/test (fold 0)"); plt.tight_layout(); plt.savefig("curves/eda_perclass.png", dpi=120); plt.show()

### Xem anh mau (3 anh / lop)

In [ ]:
from PIL import Image
fig, ax = plt.subplots(9, 3, figsize=(6, 14))
for i in range(9):
    samp = tr_df[tr_df.Label == i].head(3).Filename.tolist()
    for j, fn in enumerate(samp):
        ax[i, j].imshow(Image.open(os.path.join(IMAGES_DIR, fn)).convert("RGB"))
        ax[i, j].axis("off")
        if j == 0: ax[i, j].set_ylabel(names[i], fontsize=8)
plt.tight_layout(); plt.savefig("curves/eda_samples.png", dpi=120); plt.show()

### Kiem tra pipeline: loss ban dau ~2.197, overfit 1 batch, anh sau augmentation

In [ ]:
import torch, numpy as np
from train import set_seed
import model as M, dataset as D
set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = M.build_model("resnet50", True, 9, 0.0, "finetune").to(device)
tf_val = D.build_transforms(False, 224)
loader = D.make_loader(va_df.head(64), IMAGES_DIR, tf_val, 16, False, None, 0)
x, y, _ = next(iter(loader)); x, y = x.to(device), y.to(device)
net.eval()
with torch.inference_mode(): l0 = torch.nn.CrossEntropyLoss()(net(x), y).item()
print("loss ban dau =", round(l0,3), "(ky vong ~2.197 = -ln(1/9))")
tiny = D.make_loader(tr_df.head(8), IMAGES_DIR, D.build_transforms(True, 224, "basic"), 8, True, None, 0)
opt = torch.optim.Adam([p for p in net.parameters() if p.requires_grad], lr=1e-2)
net.train()
for it in range(40):
    xb, yb, _ = next(iter(tiny)); xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(); loss = torch.nn.CrossEntropyLoss()(net(xb), yb); loss.backward(); opt.step()
print("loss sau overfit 8 anh =", round(loss.item(),4), "(ky vong gan 0)")
mean = np.array(D.IMAGENET_MEAN).reshape(3,1,1); std = np.array(D.IMAGENET_STD).reshape(3,1,1)
fig, ax = plt.subplots(1, 4, figsize=(10,3))
xb, yb, _ = next(iter(tiny))
for j in range(4):
    img = (xb[j].numpy()*std+mean).transpose(1,2,0).clip(0,1)
    ax[j].imshow(img); ax[j].set_title(names[int(yb[j])], fontsize=8); ax[j].axis("off")
plt.tight_layout(); plt.savefig("curves/eda_aug.png", dpi=120); plt.show()

## 3. Buoc 1 — So sanh backbone (>=5, cung cong thuc nen, cung seed)

In [ ]:
FULL = False  # doi True de chay that 12 epoch
EPOCHS = 12 if FULL else 2
from train import Config, run
BACKBONES = ["resnet50","resnext50_32x4d","convnext_tiny","deit_small_patch16_224",
             "swin_tiny_patch4_window7_224","efficientnet_b0","mobilenetv3_large_100"]
rows = []
for i, bb in enumerate(BACKBONES):
    eid = "B%02d" % (i+1)
    try:
        s = run(Config(exp_id=eid, backbone=bb, seed=0, epochs=EPOCHS, batch_size=64,
                        images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
        s.update(exp_id=eid, backbone=bb); rows.append(s); print(eid, bb, "OK", round(s["val_macro_f1"],4))
    except Exception as e:
        print(eid, bb, "LOI:", type(e).__name__, str(e)[:200])
df1 = pd.DataFrame(rows); df1.to_csv("step1_summary.csv", index=False)
print(df1[["exp_id","backbone","params_M","gmacs","val_macro_f1","val_top1","train_s_per_epoch"]].to_string())

## 4. Buoc 2 — Cong thuc huan luyen (>=3 truc, moi lan khac nen 1 yeu to)

In [ ]:
BEST_BB = "resnet50"  # DOI theo ket qua Buoc 1 (F1 cao nhat / can bang voi do tre)
BASE_CFG = dict(backbone=BEST_BB, seed=0, epochs=EPOCHS, batch_size=64,
                images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
run(Config(exp_id="T00", **BASE_CFG))
EXPS = [("T01", dict(init="scratch")), ("T02", dict(init="frozen")),
        ("T03", dict(aug="color")), ("T04", dict(aug="trivial")),
        ("T05", dict(mix="cutmix", mix_alpha=1.0)),
        ("T06", dict(loss="ls", label_smoothing=0.1)),
        ("T07", dict(loss="focal", focal_gamma=2.0)),
        ("T08", dict(loss="ce_weighted")), ("T09", dict(sampler="balanced")),
        ("T10", dict(lr_backbone=1e-3)), ("T11", dict(ema_decay=0.999))]
for eid, kw in EXPS:
    try:
        run(Config(exp_id=eid, **dict(BASE_CFG, **kw)))
    except Exception as e:
        print(eid, "LOI:", type(e).__name__, str(e)[:200])
print("Xong. So sanh macro-F1 val cua T01-T11 voi T00 trong runs/*/summary.json")

## 5. Buoc 3 — Suy luan (>=4 PP) + do tre dung cach

In [ ]:
import json, torch, numpy as np
import dataset as D, model as M
from inference import predict_logits, view_hflip, views_multicrop, views_multiscale, aggregate_views, fit_temperature, apply_temperature, fuse_conv_bn
from benchmark import latency_report
from eval import compute_metrics
from torchvision import transforms as Tv
EXP = "T00"  # DOI thanh exp tot nhat Buoc 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tr_df, va_df, te_df = D.load_split(LABELS_DIR, 0)
valloader = D.make_loader(va_df, IMAGES_DIR, D.build_transforms(False, 224), 64, False, None, 2)
ckpt = torch.load("runs/" + EXP + "/seed0/best.pt", map_location=device)
cfg = ckpt["cfg"]
net = M.build_model(cfg["backbone"], False, 9, cfg.get("drop_rate", 0.0), "finetune")
net.load_state_dict(ckpt["state"]); net.to(device).eval()
names, yt, logits = predict_logits(net, valloader, device)
sm = lambda z: (lambda e: e/e.sum(1, keepdims=True))(np.exp(z-z.max(1, keepdims=True)))
base = compute_metrics(yt, logits.argmax(1), sm(logits))
print("I00 1-view: F1=%.4f top1=%.4f ECE=%.4f" % (base["macro_f1"], base["top1"], base["ece"]))
fn_f, _, logits_f = predict_logits(net, valloader, device, view_hflip)
assert fn_f == names
for space in ("prob", "logit"):
    p = aggregate_views([logits, logits_f], space)
    m = compute_metrics(yt, p.argmax(1), p)
    print("I01 flip-%s: F1=%.4f top1=%.4f (K=2, ~2x tre)" % (space, m["macro_f1"], m["top1"]))
tf256 = Tv.Compose([Tv.Resize(256), Tv.ToTensor(), Tv.Normalize(mean=list(D.IMAGENET_MEAN), std=list(D.IMAGENET_STD))])
loader256 = D.make_loader(va_df, IMAGES_DIR, tf256, 64, False, None, 2)
allp = []
with torch.inference_mode():
    for x, y, fn in loader256:
        x = x.to(device)
        ls = [net(v).detach().cpu().numpy() for v in views_multicrop(x, 224)]
        allp.append(aggregate_views(ls, "prob"))
p5 = np.concatenate(allp)
m5 = compute_metrics(yt, p5.argmax(1), p5)
print("I02 5-crop: F1=%.4f top1=%.4f (K=5, ~5x tre)" % (m5["macro_f1"], m5["top1"]))
try:
    allm = []
    with torch.inference_mode():
        for x, y, fn in loader256:
            x = x.to(device)
            ls = [net(v).detach().cpu().numpy() for v in views_multiscale(x, [224, 256, 288])]
            allm.append(aggregate_views(ls, "prob"))
    pm = np.concatenate(allm)
    mm = compute_metrics(yt, pm.argmax(1), pm)
    print("I04 multiscale 224/256/288: F1=%.4f top1=%.4f" % (mm["macro_f1"], mm["top1"]))
except Exception as e:
    print("I04 multiscale bo qua:", type(e).__name__, str(e)[:150], "(ViT/Swin can xu ly pos-embed)")
T = fit_temperature(logits, yt)
pcal = apply_temperature(logits, T)
print("I07 temperature: T=%.3f ECE truoc=%.4f sau=%.4f (acc khong doi)" % (T, base["ece"], compute_metrics(yt, pcal.argmax(1), pcal)["ece"]))
json.dump({"T": T, "EXP": EXP, "cfg": cfg}, open("temp_scaling.json", "w"))
import copy
net_bf = copy.deepcopy(net).eval()
net_fused = fuse_conv_bn(copy.deepcopy(net).eval())
xb, _, _ = next(iter(valloader)); xb = xb.to(device)
with torch.inference_mode():
    dmax = (net_bf(xb) - net_fused(xb)).abs().max().item()
print("I08 gop BN: sai so lon nhat truoc/sau = %.2e (ky vong <=1e-5; kien truc LayerNorm thi bo qua)" % dmax)
lat = {}
dev = "cuda" if torch.cuda.is_available() else "cpu"
for bs in (1, 32):
    for dt in (["fp32", "amp"] if torch.cuda.is_available() else ["fp32"]):
        r = latency_report(net, bs, 224, dt, dev)
        lat[str(bs)+"_"+dt] = r
        print("batch=%d %s: p50=%.1f p95=%.1f p99=%.1f ms (%.0f anh/s) [%s]" % (bs, dt, r["p50"], r["p95"], r["p99"], r["images_per_s"], r["gpu"]))
json.dump({"EXP": EXP, "I00": {"macro_f1": base["macro_f1"], "top1": base["top1"], "ece": base["ece"]},
           "latency": lat, "T": T}, open("inference_results.json", "w"))

## 6. Buoc 4 — Chung ket: >=3 seed, test 1 lan/seed (sau buoc nay KHONG sua cau hinh)

In [ ]:
from train import Config, run
FINAL_CFG = dict(backbone=BEST_BB, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
                 save_test_predictions=True)
FINAL_CFG.update({})  # THEM cac tuy chon thang tu Buoc 2, vd: dict(aug="trivial", loss="ls", label_smoothing=0.1, ema_decay=0.999)
for seed in (0, 1, 2):
    run(Config(exp_id="F01", seed=seed, **FINAL_CFG))
    run(Config(exp_id="T00", seed=seed, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, save_test_predictions=True))
import json, torch, numpy as np, dataset as D, model as M
from inference import predict_logits, apply_temperature
from eval import save_predictions
ts = json.load(open("temp_scaling.json"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tr_df, va_df, te_df = D.load_split(LABELS_DIR, 0)
valloader = D.make_loader(va_df, IMAGES_DIR, D.build_transforms(False, 224), 64, False, None, 2)
testloader = D.make_loader(te_df, IMAGES_DIR, D.build_transforms(False, 224), 64, False, None, 2)
sm = lambda z: (lambda e: e/e.sum(1, keepdims=True))(np.exp(z-z.max(1, keepdims=True)))
for exp in ["F01", "T00"]:
    T = ts["T"] if exp == "F01" else 1.0
    for seed in (0, 1, 2):
        ck = torch.load("runs/%s/seed%d/best.pt" % (exp, seed), map_location=device)
        c = ck["cfg"]
        n = M.build_model(c["backbone"], False, 9, c.get("drop_rate", 0.0), "finetune")
        n.load_state_dict(ck["state"]); n.to(device).eval()
        fv, yv, lv = predict_logits(n, valloader, device)
        save_predictions("predictions/%s_seed%d_val.csv" % (exp, seed), fv, yv, sm(lv))
        ft, yt2, lt = predict_logits(n, testloader, device)
        save_predictions("predictions/%suncal_seed%d_test.csv" % (exp, seed), ft, yt2, sm(lt))
        save_predictions("predictions/%s_seed%d_test.csv" % (exp, seed), ft, yt2, apply_temperature(lt, T))
        print(exp, "seed", seed, "OK")

### Diem theo dung dinh nghia cua lop (eval.py)

In [ ]:
!python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
!python eval.py score --pred "predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag T00 --out eval_out
!python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --uncal "predictions/F01uncal_seed*_test.csv" --final-val "predictions/F01_seed*_val.csv" --test-csv data/labels/test_subset0.csv --val-csv data/labels/val_subset0.csv --labels data/labels/labels.csv --out eval_out
# Neu co p95 batch-1 do dung cach: chay them lenh grade voi --latency-p95-ms <so> (lay tu Buoc 3).

## 7. Buoc 5 — Dong goi san pham + tai ve

In [ ]:
import json, glob, pandas as pd
from pathlib import Path
rows = []
for s in sorted(glob.glob("runs/*/seed*/summary.json")):
    d = json.load(open(s)); d["path"] = s; rows.append(d)
pd.DataFrame(rows).to_csv("all_runs_summary.csv", index=False)
print(pd.DataFrame(rows).to_string())
print("curves:", len(glob.glob("curves/*.png")), "| predictions:", sorted(glob.glob("predictions/*.csv")))
from google.colab import files
import shutil
shutil.make_archive("colab_outputs", "zip", ".", "predictions")
files.download("colab_outputs.zip")
files.download("all_runs_summary.csv")
print("Da tai predictions. Copy them: runs/*/summary.json, history.csv, curves/*.png, eval_out/grade_I.json vao results.xlsx + report.md (mau co san trong bai nop).")